# PD Analysis (Modular)

This notebook is a concise entry point. All heavy lifting lives in `pd_pipeline/`.


In [20]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd

from pd_pipeline import basel, capital, config, data, lasso, plots, portfolio, scenario, sensitivity


In [21]:
# Load and merge macro + GPR data
macro_frames = data.load_macro_data(
    gdp_path      = '../data/macro/global_gdp_monthly.csv',
    interest_path = '../data/macro/intrest FRED.csv',
    brent_path    = '../data/macro/brent_oil_monthly.csv',
    fuel_path     = '../data/macro/fuel_index_monthly.csv',
    cpi_path      = '../data/macro/global_cpi_monthly.csv',
    verbose       = True,
)

df_gpr = data.load_gpr_data('../data/geopolitical/data_gpr_Data_GPR.csv', verbose=True)

df_merged = data.merge_macro_data(macro_frames, df_gpr)

# Covariance / correlation on unlagged variables only (used for scenario analysis)
cov_matrix, corr_matrix, mean_vector = data.summarize_macro_data(
    df_merged,
    config.ALL_PREDICTOR_COLS,
    verbose=True,
)

# Add t-1 … t-12 lags for all macro + GPR variables
df_merged = data.add_macro_lags(df_merged, config.MACRO_COLS + config.GPR_COLS, n_lags=config.N_LAGS)
print(f"df_merged now has {df_merged.shape[1]} columns ({config.N_LAGS} lags added per variable)")

Cleaned df_gdp head:
        Date  GDP_Growth
0 2012-01-01   95.266073
1 2012-02-01   95.453594
2 2012-03-01   95.641116
3 2012-04-01   95.828638
4 2012-05-01   95.974193

Cleaned df_interest head:
        Date  Interest_Rate
0 1954-07-01           0.80
1 1954-08-01           1.22
2 1954-09-01           1.07
3 1954-10-01           0.85
4 1954-11-01           0.83

Cleaned df_brent head:
        Date  Brent_Oil
0 1992-01-01  18.156522
1 1992-02-01  18.110000
2 1992-03-01  17.659091
3 1992-04-01  19.015909
4 1992-05-01  19.980952

Cleaned df_fuel head:
        Date  Fuel_Index
0 1992-01-01   46.771404
1 1992-02-01   46.927311
2 1992-03-01   47.083218
3 1992-04-01   47.239125
4 1992-05-01   47.178468

Cleaned df_cpi head:
        Date     CPI    Asia  Americas  Europe
0 2011-01-01  80.843  79.189    84.751  85.512
1 2011-02-01  81.250  79.557    85.168  85.956
2 2011-03-01  81.726  79.710    85.901  86.571
3 2011-04-01  82.164  80.058    86.387  87.002
4 2011-05-01  82.473  80.368    86.6

In [22]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import seaborn as sns

sns.set_theme(style='white', context='notebook')
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('white')

for ax, matrix, title in zip(
    axes,
    [cov_matrix, corr_matrix],
    ['Covariance Matrix', 'Correlation Matrix'],
):
    labels = matrix.columns.tolist()
    n = len(labels)
    vals = matrix.values

    vmax = 1.0 if 'Correlation' in title else None
    vmin = -1.0 if 'Correlation' in title else None
    cmap = 'RdBu_r' if 'Correlation' in title else 'YlOrBr'

    im = ax.imshow(vals, cmap=cmap, aspect='auto', vmin=vmin, vmax=vmax)
    cb = fig.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
    cb.outline.set_linewidth(0.5)

    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_title(title, fontsize=13, fontweight='bold', pad=12)
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)

    for i in range(n):
        for j in range(n):
            v = vals[i, j]
            txt = f"{v:.2f}" if 'Correlation' in title else f"{v:.1f}"
            color = 'white' if abs(v) > (0.6 if 'Correlation' in title else 0.7 * vals.max()) else 'black'
            ax.text(j, i, txt, ha='center', va='center', fontsize=8, color=color)

plt.suptitle('Macro & GPR Variable Matrices (Unlagged)', fontsize=14, y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Bivariate normal contour plots — all pairwise macro & geopolitical variable combinations
# Contours are derived from the covariance matrix (not historical data directly).
# Shows 50 %, 90 %, 95 %, and 99 % confidence ellipses of the fitted bivariate normal.

plots.plot_normal_contours_pairwise(
    cov_matrix=cov_matrix,
    mean_vector=mean_vector,
    cols=config.MACRO_COLS + config.GPR_COLS,
)

In [23]:
# Regenerate pdsFitchData_SIC_div2.csv with correct SIC div2 sector classification
# (Re-run this cell whenever the base PD data or sector mapping changes)
data.build_sic_div2_pds_file(
    pds_path='../data/PDs/pdsFitchData.csv',
    isin_path='../data/PDs/ISIN_PD_DATA.csv',
    output_path='../data/PDs/pdsFitchData_SIC_div2.csv',
    verbose=True,
)

# Load PDs and merge with macro data
df_pds = data.load_pds_data('../data/PDs/pdsFitchData_SIC_div2.csv', verbose=True)

df_final = data.merge_pds_macro(df_pds, df_merged, verbose=True)

# Drop rows where any current or lagged predictor is missing (removes first ~12 months per series)
df_final_cleaned = data.prepare_model_data(
    df_final,
    config.ALL_PREDICTOR_COLS_WITH_LAGS,
    sector_col=config.SECTOR_COL,
    verbose=True,
)


✓ Saved 49,022 rows to ../data/PDs/pdsFitchData_SIC_div2.csv

Sector distribution:
Sector
Finance, Insurance & Real Estate        12376
Heavy Manufacturing                      6858
Utilities                                5694
Light Manufacturing                      5657
Communications                           5268
Mining & Construction                    3104
Services                                 2914
Wholesale & Retail Trade                 2831
Unassigned                               1535
Transportation                           1444
Health, Legal & Educational Services      982
Public Administration                     359

Columns in df_pds: ['Company_number', 'Date', '12_month', 'Sector', 'PDzero']

First few rows of df_pds:
       Company_number       Date  12_month          Sector  PDzero
39773      80095907.0 2012-06-01    0.0050  Communications   0.005
39774      80095907.0 2012-06-01    0.0137  Communications   0.005
39775      80095907.0 2012-06-01    0.0137  Communi

In [24]:
# Export cleaned dataset for reuse
data.export_dataframe(df_final_cleaned, output_file='df_final_cleaned.csv', verbose=True)


✓ Successfully exported dataframe to: df_final_cleaned.csv
  - Rows: 49,022
  - Columns: 38
  - File size: 17.08 MB (approximate)


In [25]:
# ── df_final_cleaned verification ────────────────────────────────────────────
print("=" * 60)
print("df_final_cleaned  –  DATA VERIFICATION")
print("=" * 60)

print(f"\nShape : {df_final_cleaned.shape[0]:,} rows × {df_final_cleaned.shape[1]} columns")
print(f"Date range : {df_final_cleaned['Date'].min().date()}  →  {df_final_cleaned['Date'].max().date()}")
print(f"Companies  : {df_final_cleaned['Company_number'].nunique():,} unique")

if 'Sector' in df_final_cleaned.columns:
    print(f"\nSector distribution:")
    print(df_final_cleaned['Sector'].value_counts().to_string())

print("\nMissing values per column:")
missing = df_final_cleaned.isnull().sum()
missing = missing[missing > 0]
print(missing.to_string() if not missing.empty else "  None")

macro_cols = ['GDP_Growth', 'Interest_Rate', 'Brent_Oil', 'CPI',
              'GPR_Global']
available_macro = [c for c in macro_cols if c in df_final_cleaned.columns]
print(f"\nMacro summary ({', '.join(available_macro)}):")
print(df_final_cleaned[available_macro].describe().round(3).to_string())

print(f"\nPD (12_month) summary:")
print(df_final_cleaned['12_month'].describe().round(6).to_string())

print(f"\nFirst 5 rows:")
display_cols = ['Date', 'Company_number'] + (['Sector'] if 'Sector' in df_final_cleaned.columns else []) + ['12_month'] + available_macro[:3]
print(df_final_cleaned[display_cols].head(5).to_string(index=False))

df_final_cleaned  –  DATA VERIFICATION

Shape : 49,022 rows × 38 columns
Date range : 2012-06-01  →  2024-07-01
Companies  : 2,304 unique

Sector distribution:
Sector
Finance, Insurance & Real Estate        12376
Heavy Manufacturing                      6858
Utilities                                5694
Light Manufacturing                      5657
Communications                           5268
Mining & Construction                    3104
Services                                 2914
Wholesale & Retail Trade                 2831
Unassigned                               1535
Transportation                           1444
Health, Legal & Educational Services      982
Public Administration                     359

Missing values per column:
12_month    12157

Macro summary (GDP_Growth, Interest_Rate, Brent_Oil, CPI, GPR_Global):
       GDP_Growth  Interest_Rate  Brent_Oil        CPI  GPR_Global
count   49022.000      49022.000  49022.000  49022.000   49022.000
mean      107.837          1.

In [26]:
# OLS sensitivity analysis (includes current + lagged macro/GPR variables)

df_sensitivities = sensitivity.run_sensitivity_analysis(
    df_final_cleaned,
    macro_cols=config.ALL_MACRO_COLS,
    gpr_cols=config.ALL_GPR_COLS,
    sector_col=config.SECTOR_COL,
    pd_maturity_cols=config.PD_MATURITY_COLS,
    pdzero_col=config.PDZERO_COL,
    verbose=True,
)

print("\n" + "="*80)
print("SENSITIVITY ANALYSIS RESULTS")
print("="*80)
print(df_sensitivities)



Processing sector: Communications (n=5268)
  ✓ 12_month: R²=0.158, N=3926

Processing sector: Utilities (n=5694)
  ✓ 12_month: R²=0.180, N=4245

Processing sector: Mining & Construction (n=3104)
  ✓ 12_month: R²=0.162, N=2347

Processing sector: Light Manufacturing (n=5657)
  ✓ 12_month: R²=0.051, N=4204

Processing sector: Heavy Manufacturing (n=6858)
  ✓ 12_month: R²=0.157, N=5433

Processing sector: Public Administration (n=359)
  ✓ 12_month: R²=0.352, N=251

Processing sector: Services (n=2914)
  ✓ 12_month: R²=0.067, N=2167

Processing sector: Health, Legal & Educational Services (n=982)
  ✓ 12_month: R²=0.277, N=720

Processing sector: Finance, Insurance & Real Estate (n=12376)
  ✓ 12_month: R²=0.103, N=9299

Processing sector: Unassigned (n=1535)
  ✓ 12_month: R²=0.448, N=1191

Processing sector: Wholesale & Retail Trade (n=2831)
  ✓ 12_month: R²=0.059, N=1949

Processing sector: Transportation (n=1444)
  ✓ 12_month: R²=0.231, N=1133

SENSITIVITY ANALYSIS RESULTS
              

In [27]:
# Sensitivity exports + tables
sensitivity.export_sensitivities(df_sensitivities, output_file='sensitivity_results_with_CI.csv')

sensitivity.print_sensitivity_tables(df_sensitivities, config.ALL_MACRO_COLS, config.ALL_GPR_COLS)

sensitivity.print_confidence_interval_summary(df_sensitivities, config.ALL_GPR_COLS)

# Uncomment for a full per-sector printout (very verbose)
# sensitivity.print_sensitivity_details(df_sensitivities, config.ALL_MACRO_COLS, config.ALL_GPR_COLS)


✓ Sensitivity results with 95% confidence intervals exported to: sensitivity_results_with_CI.csv
  Total sectors analyzed: 12

Columns include:
  - Point estimates: β_[variable] and δ_[variable]
  - 95% CI lower bounds: β_[variable]_CI_lower and δ_[variable]_CI_lower
  - 95% CI upper bounds: β_[variable]_CI_upper and δ_[variable]_CI_upper
MACRO SENSITIVITIES (β) - Impact of macroeconomic variables on PD
                                  Sector PD_Horizon  N_observations  \
0                         Communications   12_month            3926   
1                              Utilities   12_month            4245   
2                  Mining & Construction   12_month            2347   
3                    Light Manufacturing   12_month            4204   
4                    Heavy Manufacturing   12_month            5433   
5                  Public Administration   12_month             251   
6                               Services   12_month            2167   
7   Health, Legal & Educa

### Sensitivity analysis — reporting fit and coefficients

**Model fit.** Ordinary \(R^2\) never decreases when you add regressors, so it is optimistic when comparing specifications with different numbers of variables. **Adjusted \(R^2\)** penalizes extra predictors and is the standard in-sample metric for nested OLS models on the same outcome. **AIC** and **BIC** (also reported in the tables) trade off fit and complexity; lower is better, with BIC penalizing extra parameters more strongly than AIC.

**Caveat.** These are all in-sample measures. For claims about predictive performance, cross-validated \(R^2\) or a proper hold-out sample is stronger; your LASSO block already uses CV for regularization.

**Figures below:** (1) \(R^2\) vs adjusted \(R^2\) by sector and sample size vs fit; (2) forest plots of \(\beta\) (macro) and \(\delta\) (GPR) with 95% CIs; (3) heatmap of all coefficients in the full lagged specification; (4) a compact heatmap using only contemporaneous macro + GPR for readable in-line results.

In [28]:
# Clear sensitivity visualisations for the thesis (re-run the sensitivity + export cells if columns are missing)
_pd_h = config.PD_MATURITY_COLS[0]

plots.plot_sensitivity_model_fit(df_sensitivities, pd_horizon=_pd_h, figsize=(11, 6.5))

# Contemporaneous macro — easy to read in the main text
plots.plot_sensitivity_coefficient_forest(
    df_sensitivities, config.MACRO_COLS, kind="macro", pd_horizon=_pd_h
)

# All GPR terms (current + lags)
plots.plot_sensitivity_coefficient_forest(
    df_sensitivities, config.ALL_GPR_COLS, kind="gpr", pd_horizon=_pd_h
)

# Full lag structure: pattern across all sectors (appendix / large slide; cell values omitted if >18 predictors)
plots.plot_sensitivity_significance_heatmap(
    df_sensitivities, config.ALL_MACRO_COLS, config.ALL_GPR_COLS, pd_horizon=_pd_h
)

# Compact: current-quarter macro + GPR only
plots.plot_sensitivity_significance_heatmap(
    df_sensitivities, config.MACRO_COLS, config.GPR_COLS, pd_horizon=_pd_h, figsize=(9, 6)
)

# OLS forests — every regressor in the full lagged specification (macro figure, then GPR figure)
plots.plot_sensitivity_forests_all_predictors(
    df_sensitivities,
    config.ALL_MACRO_COLS,
    config.ALL_GPR_COLS,
    pd_horizon=_pd_h,
)

AttributeError: module 'pd_pipeline.plots' has no attribute 'plot_sensitivity_model_fit'

In [ ]:
# Cumulative (summed) OLS coefficients — one thesis figure per variable
# β_total = β_lag0 + β_lag1 + β_lag2 + β_lag3 + β_lag4  (same for δ / GPR)
# 95 % CI propagated in quadrature (independent-lags approximation)

_pd_h = config.PD_MATURITY_COLS[0]

plots.plot_cumulative_coefficient_forest(
    df_sensitivities,
    base_cols=config.MACRO_COLS,
    all_cols=config.ALL_MACRO_COLS,
    kind='macro',
    pd_horizon=_pd_h,
)

plots.plot_cumulative_coefficient_forest(
    df_sensitivities,
    base_cols=config.GPR_COLS,
    all_cols=config.ALL_GPR_COLS,
    kind='gpr',
    pd_horizon=_pd_h,
)

In [ ]:
# Elastic-Net feature selection (ElasticNetCV over alpha × l1_ratio grid)
df_lasso, lasso_selected_features = lasso.run_lasso_feature_selection(
    df_final_cleaned,
    macro_cols=config.ALL_MACRO_COLS,
    gpr_cols=config.ALL_GPR_COLS,
    sector_col=config.SECTOR_COL,
    pd_maturity_cols=config.PD_MATURITY_COLS,
    pdzero_col=config.PDZERO_COL,
    verbose=True,
)

feature_freq_df = lasso.print_lasso_summary(df_lasso, config.ALL_MACRO_COLS, config.ALL_GPR_COLS)
comparison_full = lasso.compare_ols_lasso(df_sensitivities, df_lasso, config.ALL_MACRO_COLS, config.ALL_GPR_COLS)
lasso.export_lasso_outputs(df_lasso, comparison_full)
lasso.print_feature_recommendations(feature_freq_df, comparison_full)

# Bootstrap stability (B=200 per sector — takes a few minutes)
df_lasso = lasso.run_bootstrap_stability(
    df_final_cleaned, df_lasso,
    macro_cols=config.ALL_MACRO_COLS,
    gpr_cols=config.ALL_GPR_COLS,
    sector_col=config.SECTOR_COL,
    pd_maturity_cols=config.PD_MATURITY_COLS,
    pdzero_col=config.PDZERO_COL,
    n_bootstrap=200,
    verbose=True,
)

# Regularization paths (used for the path plot below)
path_data = lasso.compute_regularization_paths(
    df_final_cleaned, df_lasso,
    macro_cols=config.ALL_MACRO_COLS,
    gpr_cols=config.ALL_GPR_COLS,
    sector_col=config.SECTOR_COL,
    pd_maturity_cols=config.PD_MATURITY_COLS,
    pdzero_col=config.PDZERO_COL,
    verbose=False,
)

In [ ]:
# LASSO / Elastic-Net visualizations
_pd_h = config.PD_MATURITY_COLS[0]

# 1. Summary: feature frequency, sector feature counts, selection heatmap, R² scatter
plots.plot_lasso_summary(df_lasso, feature_freq_df, config.MACRO_COLS, config.GPR_COLS, df_sensitivities)

# 2. Regularization paths — how coefficients evolve as penalty increases (top 4 sectors)
plots.plot_regularization_paths(path_data, config.ALL_MACRO_COLS, config.ALL_GPR_COLS, top_n=4)

# 3. Bootstrap stability heatmap — which features are robustly selected
plots.plot_bootstrap_stability(df_lasso, config.ALL_MACRO_COLS, config.ALL_GPR_COLS, pd_horizon=_pd_h)

# 4. Elastic-Net coefficient forests (native units, same scale as OLS)
plots.plot_lasso_forests_all_predictors(
    df_lasso, config.ALL_MACRO_COLS, config.ALL_GPR_COLS, pd_horizon=_pd_h
)

# 5. OLS vs Elastic-Net side-by-side comparison forests
plots.plot_ols_lasso_forest_comparison_all(
    df_sensitivities, df_lasso,
    config.ALL_MACRO_COLS, config.ALL_GPR_COLS,
    pd_horizon=_pd_h,
)

In [ ]:
# Heatmap: LASSO betas for macro & geopolitical variables by sector
# Coloured cells = variable selected (non-zero beta); grey = shrunk to zero.
# Colour encodes the standardised coefficient magnitude and sign.
plots.plot_lasso_beta_heatmap(
    df_lasso,
    macro_cols=config.ALL_MACRO_COLS,
    gpr_cols=config.ALL_GPR_COLS,
    pd_horizon=config.PD_MATURITY_COLS[0],
)